# Query planning in QuackWeave

**Architecture proposal grounded in QuackWeave `f45c2c1` and DuckDB 2.0.1 `8d78114e90`.** Give DuckDB a typed graph scan that it can plan alongside its other relations. Give the selected graph engine a read that it can plan and execute using its own indexes, traversal algorithms and storage. The integration decision is which SQL operations may cross that scan boundary without changing observable results.

For a standalone read, the natural delegation unit is the entire admitted graph statement. For SQL/PGQ, it is the graph pattern and required properties. DuckDB retains the surrounding relational expressions, joins, grouping, ordering and limits unless a specific rewrite establishes equivalence. The public entry points remain bare `MATCH`, `GRAPH_TABLE` and the explicit-schema `qw_query` function. [High-level architecture](quack-weave-architecture.md) · [Usage specification](quack-weave-hybrid-usage-spec.ipynb)

## What DuckDB separates

| Stage | Handoff in the tested host | Implication for QuackWeave |
|---|---|---|
| Parse and transform | `Parser::ParseQuery` produces statements and parsed expressions. PEG grammar extensions can lower new syntax into existing statement and table-reference forms. | Preserve graph source spans, variable names and parameter locations. Let DuckDB parse the enclosing SQL. |
| Bind and build a logical plan | `Planner::CreatePlan` invokes the binder. A table-function bind resolves a result schema so parents can reference typed columns by binding. | Resolve graph alias, catalog, generation, property types, parameters and output slots here. Graph registration can discover metadata. Query binding should avoid scanning graph data. |
| Optimize the logical plan | `Optimizer::Optimize` runs extension pre-hooks, built-in passes, then extension post-hooks. Built-ins include expression rewriting, filter pushdown, join ordering, column pruning, statistics propagation and join-filter pushdown. | Expose relational operators and truthful scan capabilities. The order matters. A rewrite introduced in a post-hook does not automatically receive a second run of the built-ins. |
| Create a physical plan | `PhysicalPlanGenerator::CreatePlan(LogicalGet &)` can produce a `PhysicalTableScan`. It accounts for filter columns, projected column IDs and whether the scan actually supports projection pushdown. | Retain the ordinary physical scan and implement the exact column contract. A capability flag is a promise about rows and slots. |
| Execute | `Executor::InitializeInternal` builds and schedules pipelines. Sources feed chunks to DuckDB operators, including the probe side of a hash join. | Acquire the provider reader at execution, after generation/access checks. Build-side watch keys become available at execution. Keep mutable reader and parameter state per execution. |

I used `code_symbol_search`, `code_read_symbol` and `code_callees` on the registered **duckdb** project. The project checkout and release pin differ. I checked the stage order and extension hooks against the pinned source. The optimizer call sequence is visible in function bodies even where the graph index leaves C++ call edges unresolved. [Parser](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/parser/parser.cpp#L253) · [Preparation](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/main/client_context.cpp#L317) · [Binder](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/planner.cpp#L213) · [Optimizer hooks](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/optimizer/optimizer.cpp#L592) · [Scan planning](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/execution/physical_plan/plan_get.cpp#L31) · [Pipeline construction](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/parallel/executor.cpp#L270)

### From parsing to the first graph batch

The diagram shows one successful execution. The physical planner is a separate DuckDB stage. The provider prepares or plans the graph fragment inside the final call. Its internal plan is opaque to DuckDB. The sequence check verifies an ordered trace of these messages. The live plans below exercise the actual extension.

In [ ]:
sequenceDiagram
    participant Grammar
    participant Binder
    participant Optimizer
    participant PhysicalPlanner
    participant Executor
    participant Provider
    Note over Grammar,Provider: @spec QUERY-LIFECYCLE
    Grammar->>Binder: parsed
    Note over Grammar,Binder: @message E1<br/>@from Grammar<br/>@to Binder<br/>@event parsed<br/>@order 1<br/>@when true<br/>@ensures STEP1: true
    Binder->>Optimizer: logical
    Note over Binder,Optimizer: @message E2<br/>@from Binder<br/>@to Optimizer<br/>@event logical<br/>@order 2<br/>@when true<br/>@ensures STEP2: true
    Optimizer->>PhysicalPlanner: optimized_logical
    Note over Optimizer,PhysicalPlanner: @message E3<br/>@from Optimizer<br/>@to PhysicalPlanner<br/>@event optimized_logical<br/>@order 3<br/>@when true<br/>@ensures LOGICAL_PLAN: true
    PhysicalPlanner->>Executor: physical_ready
    Note over PhysicalPlanner,Executor: @message E6<br/>@from PhysicalPlanner<br/>@to Executor<br/>@event physical_ready<br/>@order 4<br/>@when true<br/>@ensures PHYSICAL_PLAN: true
    Executor->>Provider: execute_fragment
    Note over Executor,Provider: @message E4<br/>@from Executor<br/>@to Provider<br/>@event execute_fragment<br/>@order 5<br/>@when true<br/>@ensures STEP4: true
    Provider->>Executor: return_batch
    Note over Provider,Executor: @message E5<br/>@from Provider<br/>@to Executor<br/>@event return_batch<br/>@order 6<br/>@when true<br/>@ensures STEP5: true
    Note over Grammar,Provider: @verify ORDER: prove sequence_protocol

## Where the current implementation crosses stages

| Entry or rule | What the code does today | Architectural consequence |
|---|---|---|
| Bare GQL or Cypher | `TransformMatch` carries the authored source and session settings into a table reference. Named parameters travel as values. The GQL frontend parses and lowers the supported read subset to Cypher for these providers. Cypher uses provider-specific read admission. | Keep an opaque, provider-owned read for standalone execution. Preserve result names, types, row multiplicity, NULLs and any specified ordering. List dialect and result-type coverage per provider. Ladybug has typed graph values, while the current RelateDB interface exposes supported scalar projections. |
| `GRAPH_TABLE` | `ParsedGraphTable::Lower` constructs a host `SelectNode`, collects graph properties and emits a Cypher projection. It also inspects SQL filters and join syntax to select a filtered or watchlist function. | Surface projections and SQL expressions compose well. A grammar transform choosing an execution strategy ties optimization to spelling, such as `ON` versus `USING`, and obscures equivalent queries through a view. |
| Aggregate pushdown | The pre-optimizer walks a bound aggregate/projection/get chain. It reconstructs provenance by checking generated `MATCH … RETURN …` text and aliases, rebinds a replacement `qw_query`, and checks registration generation. | Keep the bound logical rule. Give it validated property bindings so a formatting change cannot alter eligibility. Preserve the generation, quota and type guards. |
| Runtime watchlist | The post-hook recognizes an eligible optimized inner join, invokes DuckDB’s join-filter pass for the graph scan and bridges dynamic keys through local initialization. It temporarily adjusts scan flags for an identity projection. | Keep DuckDB’s join and build-side timing. Isolate this version-sensitive C++ bridge. General scan projection support must preserve reordered slots, filter-only columns and zero-column reads before the flags can be enabled broadly. |
| Provider execution | Rust `GraphQuery` binds and manages the reader. Ladybug uses its native prepared statement and Arrow batch path. RelateDB executes on a fresh connection to the registered frozen handle, materializes the native result, then batches it for DuckDB. | Keep the shared scan lifecycle; specify batch, memory and timeout behavior separately for each provider. A DuckDB chunk interface alone cannot make RelateDB stream an unbounded result. |

[Standalone transform](crates/native/grammar/standalone.cpp) · [PGQ lowering](crates/native/grammar/graph_table.cpp) · [Aggregate rule](crates/native/grammar/aggregate_pushdown.cpp) · [Runtime filter bridge](crates/native/grammar/runtime_watchlist.cpp) · [Bound scan and execution](crates/core/src/qw_query.rs) · [Ladybug](crates/native/src/ladybug.rs) · [RelateDB](crates/native/src/relate_backend.rs)

## Proposed ownership and interfaces

**1. Frontend and binding.** Keep two fragment forms. An opaque standalone fragment holds the authored dialect, text/span and parameter slots. A structured SQL/PGQ fragment holds the supported fixed pattern, variable identity, entity kind, label/type and graph-property references. Reuse the existing GQL parse representation where it applies. Each fragment binds to one provider, graph generation, ordered result schema and explicit graph-to-SQL column mapping. Preserve SQL aliases separately from case-sensitive graph property names. The binder must validate any descriptor reachable through a callable table function. A caller-supplied string or flag cannot certify a safe rewrite.

**2. One DuckDB logical scan per graph fragment.** Build on `BoundRead` and the existing table functions. Store validated projected slots, provider capabilities and optional bound predicates with the scan. Keep the bound description reusable across prepared executions. Put actual watch keys, native connections, cancellation state and batch cursors in execution-local state. A SQL join between two providers can have two independent scans; DuckDB owns that join. Distinct providers have distinct snapshots.

**3. Let DuckDB offer work to the scan.** Use the existing filter callback for eligible bound expressions, even when a predicate originated outside `GRAPH_TABLE`. Map the bound column index back to the graph property. Distinguish an optional prefilter that retains the SQL residual from exact enforcement that allows DuckDB to remove it. The Rust `rusty_sequence` example already demonstrates `pushdown_filter`, the index mapping and `accept_pushdown`. Extend projection pruning only for structured reads with safe property projections. Pruning an expression from an opaque native return list could suppress a conversion error, change distinctness or ordering. Supply row estimates independently of correctness bounds. The current watch hint of 65 is derived from a key threshold; a single key can produce thousands of graph rows.

**4. Keep graph-specific optimizer rules small.** One rule may use a DuckDB hash join’s exact keys to restrict a graph scan. Another may replace a proven scalar aggregate with a provider aggregate. The rule owns eligibility and the DuckDB residual. The provider owns translation to the backend dialect and native execution. Start with the pinned pre-/post-hook placements above. Move a rule to a different phase only after plan tests confirm which filters, bindings and projections are available there. [Bound filter example](crates/core/src/sequence.rs) · [Table-function callbacks](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/include/duckdb/function/table_function.hpp#L493) · [DuckDB built-in optimizer passes](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/optimizer/optimizer.cpp#L230)

### Pushdown decisions

| Operation | Evidence needed | DuckDB work that remains |
|---|---|---|
| Typed equality or a small integer `IN` | Direct bound property with compatible type, NULL and comparison semantics. A declared RelateDB type alone cannot rule out a stored value of another type. Respect configured execution limits. | Keep the residual unless enforcement and observable errors are equivalent. Unsupported expressions run as ordinary SQL filters. |
| Watchlist supplied by a table | Eligible inner equijoin, supported key type, exact keys from the current build side and the same graph generation. Today the bridge accepts up to 64 keys. | Retain the entire SQL join. A watchlist containing `(7),(7),(NULL)` must return each graph match for 7 twice. Oversized, missing or unusable dynamic filters fall back to the base scan. |
| Ungrouped scalar count or sum | Check the aggregate, argument types, filters, distinctness, NULL-on-empty behavior, overflow and resource settings. On RelateDB a frozen single-node or directed single-edge row bound can prove a positive-modulus sum fits its signed accumulator. | Keep grouped, unsafe or resource-limited cases in DuckDB. Preserve SQL `BIGINT` count and `HUGEINT` sum types. |
| SQL outer join, order, limit, or a second provider | A future rewrite would need separate evidence for null-extension, collation, ordering, cardinality and snapshot semantics. | Use DuckDB’s normal operators. A limit in an authored standalone graph statement remains part of that graph statement. |

[Watchlist fixture](test/relate/table_watch.test) · [Predicate fixture](test/relate/predicate_pushdown.test) · [Aggregate and fallback fixtures](test/relate/aggregate.test)

### Two statements, two delegation boundaries

With the graph selected and the dialect set to GQL or Cypher, this standalone read delegates the pattern, property predicate and return projection to the graph engine. The result column is `id`.

```sql
SESSION SET GRAPH social;
MATCH (m:Message)-[r:Message_hasCreator_Person]->(p:Person)
WHERE p.PersonId = 4398046519825
RETURN m.MessageId AS id;
```

With the benchmark's five-row DuckDB watch table, this hybrid statement lets the graph engine traverse the same pattern. DuckDB owns the join to the live table, duplicate handling, grouping, checksum and ordering. It can send eligible IDs to the graph scan once the join’s build side has them.

```sql
SELECT g.author, count(*), sum(g.mid % 1000003)
FROM GRAPH_TABLE (
    social MATCH (m:Message)-[r:Message_hasCreator_Person]->(p:Person)
    COLUMNS (p.PersonId AS author, m.MessageId AS mid)
) AS g
JOIN watch USING (author)
GROUP BY g.author
ORDER BY g.author;
```

GQL, Cypher and SQL/PGQ have separate language contracts. Keep a feature matrix for the accepted subset and result shapes of each backend. Reject an unsupported construct explicitly. A future provider that accepts the required GQL semantics directly can take the original GQL read after admission; the two present adapters use the supported GQL-to-Cypher lowering.

## Live plan and result checks

The next cell loads the local combined extension into the benchmark’s pinned DuckDB 2.0.1 through ADBC and uses the existing modulus-256 fixture, with 14,883 creator edges. It starts each provider independently, executes each statement, compares the result with the CSV reference, and inspects the physical graph scan. For standalone syntax it uses `PREPARE … AS MATCH …` followed by `EXPLAIN (ANALYZE, FORMAT JSON) EXECUTE …`. The pinned grammar rejects the direct `EXPLAIN … MATCH` spelling. The “selected DuckDB operators” column lists operators relevant to the boundary; the stored plan also contains execution wrappers.

The final case sets a 1 MiB conversion budget. The identical SQL then takes the ordinary graph scan and host aggregate path. Both numeric results must match. To replay on a fresh checkout, provision the combined extension, ADBC environment and modulus-256 fixture as described in the benchmark instructions. [Fixture and setup](benchmarks/provider_comparison/README.md) · [Workload SQL and reference](benchmarks/provider_comparison/run.py)

In [3]:
from pathlib import Path
import json
import subprocess
from IPython.display import Markdown, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "benchmarks/provider_comparison/run.py").is_file())
# The repository's ADBC worker loads its pinned DuckDB 2.0.1 library and enforces the source ID.
worker = "from pathlib import Path\nfrom types import SimpleNamespace\nimport json\nimport sys\n\nroot = Path.cwd()\nsys.path.insert(0, str(root / \"benchmarks\" / \"provider_comparison\"))\nimport run\n\nfixture = run.BUILD / \"mod256\"\nreference = run.oracle(fixture)\n\ndef operators(nodes):\n    for node in nodes:\n        yield node\n        yield from operators(node.get(\"children\", []))\n\nrows = []\nfor provider in run.ENGINES:\n    args = SimpleNamespace(provider=provider, fixture=fixture, extension=run.EXT, library=run.LIB)\n    with run.cursor(args) as (cur, _):\n        for case in (\"gql_lookup\", \"cypher_fanout\", \"graph_table_where\", \"table_watchlist\", \"full_scan_aggregate\", \"budget_fallback\"):\n            key = \"full_scan_aggregate\" if case == \"budget_fallback\" else case\n            dialect, sql = run.CASES[key]\n            cur.execute(\"SET SESSION qw_graph_dialect = \" + run.sql_literal(dialect))\n            if case == \"budget_fallback\":\n                config = {\"alias\": \"social\", \"path\": str((fixture / run.ENGINES[provider]).resolve()),\n                          \"max_batch_bytes\": 1048576}\n                cur.execute(f\"SET GLOBAL qw_{provider}_config = \" + run.sql_literal(json.dumps(config)))\n            # PREPARE accepts the standalone grammar even when EXPLAIN expects a SQL statement.\n            cur.execute(\"PREPARE inspect_graph AS \" + sql)\n            cur.execute(\"EXPLAIN (ANALYZE, FORMAT JSON) EXECUTE inspect_graph\")\n            plan = list(operators(json.loads(cur.fetchone()[1])[\"operator\"]))\n            answer, _ = run.fetch(cur, \"EXECUTE inspect_graph\")\n            assert answer == reference[\"expected\"][key], (provider, case, answer)\n            scan, = [node for node in plan if node[\"name\"].startswith(\"QW_\")]\n            expected_rows = {\"gql_lookup\": 1, \"cypher_fanout\": reference[\"fanout\"],\n                             \"graph_table_where\": reference[\"fanout\"],\n                             \"table_watchlist\": sum(row[1] for row in reference[\"watchlist_groups\"]),\n                             \"full_scan_aggregate\": 1, \"budget_fallback\": reference[\"edges\"]}[case]\n            assert scan[\"intermediate_rows\"] == expected_rows\n            names = [node[\"name\"] for node in plan]\n            if case == \"table_watchlist\":\n                assert \"HASH_JOIN\" in names and \"HASH_GROUP_BY\" in names\n            if case == \"budget_fallback\":\n                assert \"UNGROUPED_AGGREGATE\" in names\n            rows.append([provider, case, scan[\"name\"], expected_rows,\n                         \", \".join(n for n in names if n in (\"HASH_JOIN\", \"HASH_GROUP_BY\", \"UNGROUPED_AGGREGATE\", \"FILTER\", \"ORDER_BY\")) or \"—\"])\nprint(json.dumps(rows))"
completed = subprocess.run([str(root / "build/usage-client-venv/bin/python"), "-B", "-c", worker],
                           cwd=root, capture_output=True, text=True)
if completed.returncode:
    raise RuntimeError(completed.stderr)
rows = json.loads(completed.stdout)
assert len(rows) == 12
display(Markdown("| Provider | Query | Graph scan | Rows from graph | Selected DuckDB operators |\n"
                 "|---|---|---|---:|---|\n" +
                 "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))

| Provider | Query | Graph scan | Rows from graph | Selected DuckDB operators |
|---|---|---|---:|---|
| ladybug | gql_lookup | QW_GRAPH_QUERY | 1 | — |
| ladybug | cypher_fanout | QW_GRAPH_QUERY | 22 | — |
| ladybug | graph_table_where | QW_GRAPH_QUERY | 22 | FILTER |
| ladybug | table_watchlist | QW_GRAPH_QUERY | 82 | ORDER_BY, HASH_GROUP_BY, HASH_JOIN |
| ladybug | full_scan_aggregate | QW_QUERY | 1 | — |
| ladybug | budget_fallback | QW_GRAPH_QUERY | 14883 | UNGROUPED_AGGREGATE |
| relate | gql_lookup | QW_GRAPH_QUERY | 1 | — |
| relate | cypher_fanout | QW_GRAPH_QUERY | 22 | — |
| relate | graph_table_where | QW_GRAPH_QUERY | 22 | FILTER |
| relate | table_watchlist | QW_GRAPH_QUERY | 82 | ORDER_BY, HASH_GROUP_BY, HASH_JOIN |
| relate | full_scan_aggregate | QW_QUERY | 1 | — |
| relate | budget_fallback | QW_GRAPH_QUERY | 14883 | UNGROUPED_AGGREGATE |

## Migration and acceptance

1. **Introduce bound fragment metadata behind the existing public entry points.** Keep the current execution path as the reference. Check identical types and rows for bare GQL, bare Cypher, `GRAPH_TABLE` and explicit schema reads on both providers. Cover parameters, escaped literals, graph property case, alias collisions, same-named node and edge types, and generation changes between prepare and execute.
2. **Move static predicate selection from syntax matching to bound scan callbacks.** Compare inner and outer SQL spellings and a predicate through a view. Assert equal rows and errors with pushdown enabled and bypassed; check provider rows separately. Add projection mapping tests for reordered, repeated, unused, filter-only and zero projected columns.
3. **Migrate runtime-watch and aggregate rules one at a time.** Keep each old execution path until its replacement passes SQLLogic for `ON`/`USING`, duplicates, NULLs, empty and oversized lists, outer joins, wide sums, explicit budgets and unsupported path shapes. Assert retained host operators as well as provider row counts. Remove the redundant grammar/text recognizer only once equivalent forms are covered.
4. **Expose and benchmark the physical boundary.** Show the provider, delegation decision, retained residual and reason for fallback in plan diagnostics without logging parameter values. Benchmark cold registration, warm p50/p95, peak RSS and rows crossing the graph scan. The full SF1 baseline for the optimized release is 6,465 rows for equality, 23,316 for the table watchlist and one for the scalar aggregate on both providers. Keep this cardinality check distinct from latency and run old/new binaries on the same host. [Cross-provider SQLLogic](test/relate/benchmark_contract.test) · [Recorded SF1 observations](benchmarks/provider_comparison/optimized/mod1.json)

**First implementation slice:** carry one bound integer property from a SQL/PGQ pattern into a DuckDB filter callback; demonstrate identical results and reduced provider rows for a graph-local predicate and an outer SQL predicate. This exercises the central interface without changing the native standalone query path.

## Solver check of the proposed routing policy

Jev 1.14.0 selected `configuration.requires_any` in verify mode for the dependency between an optional watch prefilter and the retained SQL join (open gate, weakest confidence 0.75). I added five explicit dependencies from the same catalog rule. A watch prefilter also requires a verified key and a bound generation; a graph scan requires native traversal; an aggregate replacement requires equivalent aggregate semantics and permission under the configured limits. Each dependency is a separate rule, so all applicable requirements must hold.

| Declared configuration | Z3 result | Persisted solve |
|---|---|---|
| Standalone graph scan with native traversal | SAT, pass | `sol_dd486183bbb84ad7` |
| Graph scan with watch prefilter and SQL join | SAT, pass | `sol_2764ebbd6e7d45ae` |
| Graph scan with a semantically verified, allowed aggregate | SAT, pass | `sol_793ce7de664c4238` |
| Same watch prefilter with the host join removed | UNSAT, fail | `sol_e7fc447839724de8` |
| Aggregate replacement without semantic equivalence | UNSAT, fail | `sol_c6272d15a35f4495` |

These results verify the declared ownership policy. The live experiment establishes the operator placement for the tested statements. SQLLogic and differential benchmarks remain the acceptance evidence for each future rewrite.

## Implementation and verification

Standalone GQL and Cypher keep their native read path. For a hybrid query, the grammar supplies a versioned description of the graph pattern. The binder checks it against the native read and records the property behind each DuckDB output column. DuckDB can then offer projected column IDs, bound integer predicates and execution-local hash-join keys. The provider traverses the graph. DuckDB keeps the SQL filter and join. A scalar aggregate moves only when the rule verifies its type, arithmetic and resource limits.

| Task | Commit | Fixture-backed evidence |
|---|---|---|
| Bind the graph fragment | `60df489` | Validate slot order, generated aliases and the native read before planning a rewrite. |
| Project requested properties | `2e45282` | Reordered, repeated, unused, filter-only and constant projections retain their result types. |
| Place scalar aggregates | `ab212ed` | A matching authored query keeps its host aggregate. The structured PGQ form can return one aggregate row. |
| Apply static SQL predicates | `fa2cc0a` | A filter in graph syntax, outer SQL or a view reaches the same bound integer slot. DuckDB retains the residual. |
| Explain the scan | `37c350f` | The graph scan identifies its provider, operation, residual or fallback when applicable. DuckDB can separately include submitted SQL in the profile. |
| Estimate the graph scan | `57f3ec9` | Fixed vertex and one-step scans use registered populations. RelateDB's global counts can exceed a label's actual row count. |
| Read join keys at execution | `d1541e6` | DuckDB can filter the graph through a renamed view when it builds the hash table on the watchlist. The same SQL bag is returned when the join filter is disabled. |
| Preserve explicit limits for legacy watchlists | `a6472d5` | A nonmatching literal or runtime key still raises the configured batch error. A timeout-limited graph reads the original input. |
| Match Ladybug's catalog casing | `4e3d948` | A mixed-case one-step Ladybug scan receives the same estimate as the catalog spelling. RelateDB keeps exact-case binding. |
| Explain a host-owned comparison join | `1ef6b47` | A preserved outer-join input reports the retained SQL join and the absence of an eligible runtime filter. |

The live DuckDB cell above was replayed on the combined production extension built from `1ef6b47`. Its smaller fixture yielded 82 watchlist rows for each provider. The delegated scalar aggregate returned one row. An explicit conversion budget retained all 14,883 input rows for the DuckDB aggregate. The matched full-size SF1 report measures the same boundary on 3,809,607 edges.

### Per-task solver records

The identifiers below address the persisted Z3 checks. For the runtime watch, retaining the SQL join passed on both revisions. Its failing SQLLogic case measured excess rows crossing the provider boundary.

| Task | Before | After |
|---|---|---|
| Bound fragments | UNSAT `sol_1838ace3c38a434d` | SAT `sol_52d736ff7e5a4b8c` |
| Projection | UNSAT `sol_f10af08e11014c30` | SAT `sol_58bb52fa25fe4257` |
| Scalar aggregate | UNSAT `sol_972bec2f966a4c54` | SAT `sol_fb291d05b6e04f7f` |
| Static SQL predicate | UNSAT `sol_4f606a2239614bbd` | SAT `sol_4749f88d0cd949f9` |
| Runtime watch | SAT `sol_1018e65b109c415c` | SAT `sol_b027a35e71474ba9` |
| Scan diagnostics | UNSAT `sol_012830305eb04f4c` | SAT `sol_60cfb1d50ecc4cff` |
| Graph population estimates | UNSAT `sol_fbae63dddf8e4607` | SAT `sol_9451d92a74bd490a` |
| Legacy watchlist budgets | UNSAT `sol_6186d96a9b5d4137` | SAT `sol_a12e6584c3ef4faf` |
| Mixed-case Ladybug estimates | UNSAT `sol_d97f1ae23e034849` | SAT `sol_35c8f0ac19484361` |
| Ordinary host-join diagnostic | UNSAT `sol_a7ba04785c594076` | SAT `sol_9f5f1ea3e65342b0` |
| Cross-provider acceptance | UNSAT `sol_32505f7ae0f34c72` | SAT `sol_ca7bfd55763641d3` |

The tasks ran Jev before and after the change. For the fragment post-check, aggregate checks, host-join diagnostic checks and some acceptance checks, Jev's confidence gate blocked automatic routing. Those gates stayed blocked. The reviewed catalog constraints were checked separately with Z3. The solver checks verify the declared planning invariants. The fixture-backed SQLLogic cases check the actual DuckDB and provider behavior. [Cross-provider SQLLogic cases](test/relate) · [Ladybug stabilization cases](test/stabilization/sqllogic) · [Matched SF1 measurements and reproduction](benchmarks/provider_comparison/planning/README.md)